# 🎸 Empório da Música — Análise Exploratória e Pipeline de ETL
**Desafio Técnico AI Engineer — Artefact**

Este notebook documenta o processo de exploração, validação de integridade e transformação (ETL) dos dados fornecidos para a loja **Empório da Música** em Campo Grande/MS. O objetivo é higienizar os dados brutos e carregá-los em um banco relacional **SQLite** otimizado para o consumo determinístico pelo Agente de IA.

In [ ]:
import os
import json
import sqlite3
from pathlib import Path
import pandas as pd

# Caminho base dos arquivos brutos e banco gerado
RAW_DIR = Path("../arquivos")
DB_PATH = Path("../data/emporio_da_musica.db")

print("Ambiente pronto. Arquivos brutos em:", RAW_DIR.resolve())

## 1. Carregamento dos Datasets Brutos
Vamos inspecionar a volumetria e as dimensões de cada arquivo CSV fornecido.

In [ ]:
files = {
    "categories": RAW_DIR / "desafio_tecnico_ai_eng - categories.csv",
    "customers": RAW_DIR / "desafio_tecnico_ai_eng - customers.csv",
    "products": RAW_DIR / "desafio_tecnico_ai_eng - products.csv",
    "promotions": RAW_DIR / "desafio_tecnico_ai_eng - promotions.csv",
    "orders": RAW_DIR / "desafio_tecnico_ai_eng - orders.csv",
    "order_items": RAW_DIR / "desafio_tecnico_ai_eng - order_items.csv",
}

dfs = {k: pd.read_csv(v, encoding="utf-8") for k, v in files.items()}
for name, df in dfs.items():
    print(f"Dataset: {name:12} | Linhas: {df.shape[0]:2} | Colunas: {df.shape[1]:2}")

## 2. Análise Exploratória e Diagnóstico de Qualidade
### 2.1 Clientes e Anomalia de Duplicidade
Identificamos que na base de clientes existe e-mail duplicado:

In [ ]:
df_cust = dfs["customers"]
dups = df_cust[df_cust.duplicated(subset=["email"], keep=False)]
dups

> **Decisão de Engenharia:** Bruno Martins possui dois cadastros (`customer_id` 11 e 43). O pedido #10 referencia o cliente 11. Para não perder rastreabilidade transacional nem descartar o registro 43, o campo `email` é indexado sem `UNIQUE` no banco SQLite, permitindo ao agente localizar o histórico completo.

### 2.2 Produtos e Especificações Técnicas (JSON)
A coluna `specs` armazena especificações técnicas em JSON stringificado. Validamos a integridade do JSON:

In [ ]:
df_prod = dfs["products"]
print("Faixa de preços:", df_prod["price_brl"].min(), "até", df_prod["price_brl"].max())
print("Distribuição de estoque:\n", df_prod["stock_quantity"].describe())

# Exemplo de specs técnica parseada
sample_spec = json.loads(df_prod.iloc[0]["specs"])
print("\nExemplo de especificação de produto (ID 81 - Yamaha C40):")
print(json.dumps(sample_spec, indent=2, ensure_ascii=False))

### 2.3 Pedidos e Rastreamento
Análise da correlação entre status do pedido e código de rastreamento:

In [ ]:
df_orders = dfs["orders"]
print(df_orders["status"].value_counts())
df_orders[["order_id", "status", "tracking_code", "estimated_delivery", "notes"]].head(10)

Pedidos com status `delivered` e `shipped` possuem tracking e previsão, enquanto pedidos `pending`, `confirmed` e `cancelled` mantêm `null` coerentemente.

## 3. Validação dos Schemas com Pydantic
Os schemas foram modelados em `src/models/schemas.py` com validações rigorosas de negócio.

In [ ]:
import sys
sys.path.append("..")
from src.models.schemas import Product, Customer, Order

# Validando o primeiro produto via schema Pydantic
p = Product(**dfs["products"].iloc[0].to_dict())
print("Produto validado:", p.name, "- R$", p.price_brl)
print("Specs parseado como dicionário:", type(p.specs))

## 4. Execução do ETL e Carga no SQLite
O script modular `src/etl/clean_and_load.py` executa todo o pipeline e popula o banco de dados relacional.

In [ ]:
from src.etl.clean_and_load import run_etl
counts = run_etl()
print("Carga realizada com sucesso:", counts)

## 5. Auditoria de Integridade Referencial no SQLite
Executamos a verificação de integridade de Foreign Keys no banco gerado:

In [ ]:
conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON;")
fk_violations = conn.execute("PRAGMA foreign_key_check;").fetchall()
print("Violações de integridade referencial:", len(fk_violations))
assert len(fk_violations) == 0, "Erro: existem chaves estrangeiras órfãs!"

# Query de exemplo: Top 5 produtos disponíveis em estoque
query = """
SELECT p.product_id, p.name, c.name as categoria, p.price_brl, p.stock_quantity
FROM products p
JOIN categories c ON p.category_id = c.category_id
WHERE p.status = 'active' AND p.stock_quantity > 0
ORDER BY p.price_brl DESC
LIMIT 5;
"""
pd.read_sql_query(query, conn)

## Conclusão
- Todos os datasets foram explorados, higienizados e validados com Pydantic.
- O banco `data/emporio_da_musica.db` foi criado com sucesso, com índices e foreign keys íntegras.
- As ferramentas do Agente (`FastMCP`) agora têm uma base de dados relacional rápida, consistente e determinística para consultar estoque, preços e pedidos.